# 移動表現を分類する
第7回の発展実習です。コードやエラーについてはAIに質問できます。本文の読み、設定の選択、その理由は自分で考えてください。

In [ ]:
#@title 準備（実行のみ）
import json
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.linear_model import LogisticRegression


url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/class7-consensus-v1.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = [json.loads(line) for line in bundle.open("passages.jsonl")]
    ballots = [json.loads(line) for line in bundle.open("llm-labels.jsonl")]
    reader = json.load(bundle.open("reader-labels.json"))
    with np.load(bundle.open("embeddings.npz")) as data:
        vectors = dict(zip(data["ids"], data["vectors"].astype("float32")))
items = {p["id"]: p for p in passages}
train = [b for b in ballots if items[b["id"]]["split"] == "train" and b["label"]]
validation = [b for b in ballots if items[b["id"]]["split"] == "dev" and b["label"]]
valid = [b for b in validation if items[b["id"]].get("sampling") != "negation-cue"]
targeted = [b for b in validation if items[b["id"]].get("sampling") == "negation-cue"]
train_x = [vectors[b["id"]] for b in train]
train_y = [b["label"] for b in train]

cohorts = {'random': '無作為抽出', 'rare-9b': '旧Qwenの少数ラベル'}

def show_passage(i):
    p = items[i]
    print(p["author"], p["title"], i)
    print(p["previous"])
    print(p["left"] + "【" + p["target"] + "】" + p["right"])
    print(p["next"])

def validation_scores(sample):
    ids = [b["id"] for b in sample]
    predictions = model.predict([vectors[i] for i in ids])
    hits = guesses = reference = agreements = 0
    for row, prediction in zip(sample, predictions):
        gold = row["label"]
        agreements += prediction == gold
        reference += gold == "CT-"
        guesses += prediction == "CT-"
        hits += gold == prediction == "CT-"
    def ratio(numerator, denominator):
        if denominator:
            return f"{numerator} / {denominator}"
        return '—（分母が0）'

    print('一致', ratio(agreements, len(sample)))
    print('CT−再現率', ratio(hits, reference))
    print('CT−適合率', ratio(hits, guesses))
    print('余分なCT−予測', guesses - hits)
    for label in ("CT+", "CT-", "Uu", "NM"):
        actual = [pred for row, pred in zip(sample, predictions) if row["label"] == label]
        if actual:
            print(label, 'の再現率', ratio(actual.count(label), len(actual)))

def reveal_reading(i):
    print('読者', reader[i]["label"], reader[i]["evidence"])
    print('分類器の予測', model.predict([vectors[i]])[0])

def reader_scores():
    first_disagreement = None
    for cohort, name in cohorts.items():
        ids = [i for i, r in reader.items() if r["cohort"] == cohort and r["label"] != "defer"]
        agreements = 0
        for i, pred in zip(ids, model.predict([vectors[i] for i in ids])):
            if reader[i]["label"] == pred:
                agreements += 1
            elif first_disagreement is None:
                first_disagreement = i
        print(name, agreements, "/", len(ids))
    return first_disagreement

case = next(b for b in train if b["agreement"] == 2)
reading_ids = [b["id"] for b in targeted if b["label"] == "CT-"]
print('準備完了')

## 1. 学習例を一つ読む
対象語が上下移動を表す場合、起きたならCT+、起きなかったならCT-、成否が示されなければUuです。移動以外ならNMです。本文の語句から自分のラベルを決めてから、次のセルで回答を見てください。

In [ ]:
show_passage(case["id"])

### 3モデルの回答を見る

In [ ]:
for name, vote in case["votes"].items():
    if vote:
        print(name, vote["label"], vote["evidence"])
print('多数決の仮ラベル', case["label"])

## 2. 四つの検証例を読む
動詞の前後で、移動が起きたかどうかを示す語句を探してください。

In [ ]:
for passage_id in reading_ids:
    show_passage(passage_id)

最初は標準の重みで実行します。次に下のセルの None を "balanced" に変え、学習と集計をもう一度実行してください。拾えたCT−と、余分にCT−と予測した箇所を比べます。

In [ ]:
weights = None

In [ ]:
model = LogisticRegression(class_weight=weights, max_iter=2000)
model.fit(train_x, train_y)
print('元の検証用')
validation_scores(valid)
print('否定表現から追加した検証用')
validation_scores(targeted)

## 3. 読者の判断と比べる
設定とその理由をメモしてから実行します。20作品を学習から外しました。無作為抽出12件のうち1件は対象語の範囲に問題があり保留です。旧Qwenの少数ラベルから選んだ8件は別に数えます。

In [ ]:
example = reader_scores()
show_passage(example)

自分ならどのラベルを付けますか。判断の手がかりを一つ挙げてから、次のセルを実行してください。

In [ ]:
reveal_reading(example)